In [ ]:
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4.1")
model

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    genre: str = Field(description="The genre of the movie") # these description are useful which will be used toplace the exact outputs that are coming form the LLM.
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10")

In [ ]:
model_with_structured_output = model.with_structured_output(Movie)
print(model_with_structured_output)
model_with_structured_output.invoke("Tell me about the movie Inception.")
# pydantic does the field validation and parsing of the output from the model into the Movie class. If the model's output does not match the expected structure, pydantic will raise a validation error.

## Message Output along with Parsed Structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A Movie with details"""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    genre: str = Field(..., description="The genre of the movie") # these description are useful which will be used toplace the exact outputs that are coming form the LLM.
    rating: float = Field(..., description="The rating of the movie on a scale of 1 to 10")

model_with_structured_output = model.with_structured_output(Movie, include_raw=True) # include_raw - initial raw message from the model will be included in the output. This is useful for debugging and understanding how the model arrived at its final output.

## Nested Structure

In [ ]:
class Actor(BaseModel):
    name: str
    role: str

class MovieWithActors(BaseModel):
    title: str
    year: int
    genre: str
    rating: float
    cast: list[Actor]
    budget: float | None = Field(default=None, description="The budget of the movie in USD") # optional field with default value None
    

## TypedDict

In [ ]:
from typing_extensions import Annotated, TypedDict

class Movie(TypedDict):
    title: Annotated[str, ..., "the title of the movie"]
    year: Annotated[int, ..., "the year the movie was released"]
    genre: Annotated[str, ..., "the genre of the movie"]
    rating: Annotated[float, ..., "the rating of the movie on a scale of 1 to 10"]

model_with_typeddict = model.with_structured_output(Movie)


In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieWithActors(TypedDict):
    title: str
    year: int
    genre: str
    rating: float
    cast: list[Actor]
    budget: float | None = Field(default=None, description="The budget of the movie in USD") # optional field with default value None

model_with_typeddict = model.with_structured_output(MovieWithActors)
    

## DataClasses

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class Contact(BaseModel):
    """Contact information for a person or organization"""
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model='gpt-4.1',
    response_format=Contact
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is the contact information for John Doe?"
    }]
})

print(result["structured_response"])

In [ ]:
# from pydantic import BaseModel, Field
# Using TypedDict for structured output were there is no field validation
from typing_extensions import Annotated, TypedDict
from langchain.agents import create_agent

class Contact(TypedDict):
    """Contact information for a person or organization"""
    name: str 
    email: str 
    phone: str 

agent = create_agent(
    model='gpt-4.1',
    response_format=Contact
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is the contact information for John Doe?"
    }]
})

print(result["structured_response"])

In [ ]:
# from pydantic import BaseModel, Field
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Contact:
    """Contact information for a person or organization"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model='gpt-4.1',
    response_format=Contact
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is the contact information for John Doe?"
    }]
})

print(result["structured_response"])